In [ ]:
import dlib
from PIL import Image
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
import os

In [ ]:
import zipfile

path = '/content/drive/MyDrive/Visão Computacional Guia Completo/Datasets/yalefaces.zip'
zip_objetic = zipfile.ZipFile(file=path, mode='r')
zip_objetic.extractall('./')
zip_objetic.close

<bound method ZipFile.close of <zipfile.ZipFile filename='/content/drive/MyDrive/Visão Computacional Guia Completo/Datasets/yalefaces.zip' mode='r'>>

In [ ]:
detector_facial = dlib.get_frontal_face_detector()
detector_pontos = dlib.shape_predictor('/content/drive/MyDrive/Visão Computacional Guia Completo/Weights/shape_predictor_68_face_landmarks.dat')

In [ ]:
imagem = cv2.imread('/content/drive/MyDrive/Visão Computacional Guia Completo/Images/people2.jpg')
deteccoes = detector_facial(imagem, 1)
for face in deteccoes:
  pontos = detector_pontos(imagem, face)
  for ponto in pontos.parts():
    cv2.circle(imagem, (ponto.x, ponto.y), 2, (0, 255, 0), 1)
  l, t, r, b = face.left(), face.top(), face.right(), face.bottom()
  cv2.rectangle(imagem, (l,t), (r,b), (0, 255, 255), 2)

cv2_imshow(imagem)

In [ ]:
descritor_facial_extrator = dlib.face_recognition_model_v1('/content/drive/MyDrive/Visão Computacional Guia Completo/Weights/dlib_face_recognition_resnet_model_v1.dat')

In [ ]:
index = {}
idx = 0
descritores_faciais = None

paths = [os.path.join('/content/yalefaces/train', f) for f in os.listdir('/content/yalefaces/train')]

for path in paths:
  imagem = Image.open(path).convert('RGB')
  imagem_np = np.array(imagem, 'uint8')
  deteccoes = detector_facial(imagem_np, 1)

  for face in deteccoes:
    l, t, r, b = face.left(), face.top(), face.right(), face.bottom()
    cv2.rectangle(imagem_np, (l, t), (r, b), (0, 0, 255), 2)

    pontos = detector_pontos(imagem_np, face)
    for ponto in pontos.parts():
      cv2.circle(imagem_np, (ponto.x, ponto.y), 2, (0, 255, 0), 1)

    descritor_facial = descritor_facial_extrator.compute_face_descriptor(imagem_np, pontos)
    descritor_facial = [f for f in descritor_facial]
    descritor_facial = np.asarray(descritor_facial, dtype=np.float64)
    descritor_facial = descritor_facial[np.newaxis, :]

    if descritores_faciais is None:
      descritores_faciais = descritor_facial
    else:
      descritores_faciais = np.concatenate((descritores_faciais, descritor_facial), axis = 0)
    index[idx] = path
    idx += 1

In [ ]:
descritores_faciais.shape

(132, 128)

In [ ]:
descritores_faciais

array([[-0.11487487,  0.08746712,  0.1005666 , ...,  0.00096075,
         0.04400125,  0.03493588],
       [-0.15216462,  0.06840788,  0.10447602, ..., -0.01414922,
         0.12608844,  0.03556602],
       [-0.12717085,  0.09929401,  0.06757956, ...,  0.01595355,
         0.12304637,  0.00980245],
       ...,
       [-0.13317478,  0.02902658,  0.03295897, ...,  0.02070315,
         0.06136051,  0.07010031],
       [-0.15520753,  0.12741271,  0.05637256, ...,  0.01137642,
         0.09131449, -0.0057907 ],
       [-0.08667115,  0.06839487,  0.03028989, ...,  0.02293231,
         0.11814547,  0.04629448]])

In [ ]:
len(index)

132

In [ ]:
descritores_faciais[131]

array([-0.08667115,  0.06839487,  0.03028989, -0.03679525, -0.12073934,
       -0.06566143, -0.06806837, -0.13256565,  0.10222015, -0.1102905 ,
        0.17137988, -0.0806508 , -0.15116642, -0.06908116, -0.02626503,
        0.18002528, -0.204676  , -0.15326017, -0.07769444, -0.00240113,
        0.05711802, -0.00103748,  0.01643984,  0.00492222, -0.09745561,
       -0.3371852 , -0.1096995 , -0.08994249,  0.0447119 , -0.05661511,
       -0.0875762 ,  0.04667124, -0.1934256 , -0.10386962,  0.02577351,
        0.07156256, -0.00294232, -0.0672181 ,  0.19114858, -0.01430293,
       -0.22695588,  0.03192666,  0.07998642,  0.21584079,  0.24870437,
        0.02845487,  0.0244259 , -0.10213973,  0.08417494, -0.14534158,
        0.02887923,  0.10391001,  0.06280403,  0.07917204, -0.01650877,
       -0.10575479,  0.03299883,  0.07384749, -0.12838855,  0.00702674,
        0.12304334, -0.12882316, -0.05388968, -0.04176186,  0.21134005,
        0.05466437, -0.05499198, -0.19311473,  0.11232698, -0.14

In [ ]:
confianca = 0.5
paths = [os.path.join('/content/yalefaces/test', f) for f in os.listdir('/content/yalefaces/test')]
previsoes = []
saidas_esperadas = []


for path in paths:
  imagem = Image.open(path).convert('RGB')
  imagem_np = np.array(imagem, 'uint8')
  deteccoes = detector_facial(imagem_np, 1)

  for face in deteccoes:
    pontos = detector_pontos(imagem_np, face)
    descritor_facial = descritor_facial_extrator.compute_face_descriptor(imagem_np, pontos)
    descritor_facial = [f for f in descritor_facial]
    descritor_facial = np.asarray(descritor_facial, dtype=np.float64)
    descritor_facial = descritor_facial[np.newaxis, :]

    distancia = np.linalg.norm(descritor_facial - descritores_faciais, axis=1)
    indice_min = np.argmin(distancia)
    distancia_min = distancia[indice_min]
    if distancia_min <= confianca:
      nome_previsao = int(os.path.split(index[indice_min])[1].split('.')[0].replace('subject', ''))
    else:
      nome_previsao = 'Face não identificada'

    nome_real = int(os.path.split(path)[1].split('.')[0].replace('subject', ''))

    previsoes.append(nome_previsao)
    saidas_esperadas.append(nome_real)

    cv2.putText(imagem_np, 'Pred:' + str(nome_previsao), (10, 30), cv2.FONT_HERSHEY_COMPLEX_SMALL, 1, (0, 255, 0))
    cv2.putText(imagem_np, 'Exp:' + str(nome_real), (10, 50), cv2.FONT_HERSHEY_COMPLEX_SMALL, 1, (0, 255, 0))

  cv2_imshow(imagem_np)

previsoes = np.array(previsoes)
saidas_esperadas = np.array(saidas_esperadas)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(saidas_esperadas, previsoes)

1.0